In this data, we want to understand customer behavior, try patterns, traffic congestion, airport demand, weather effects, and revenue trends in nyc. 

In [9]:
# import packages
import os  # directory management 
import pandas as pd
import pyarrow.parquet as pq #import parquet data
from pathlib import Path #import path management

 

In [11]:
# add the src folder to the path so that we can import modules from it
import sys # import sys module

#sys.path.append(os.path.join(os.path.dirname(__file__), "..", "src"))
sys.path.append("../src") # relative path to src

#print(sys.path) # print the path to check if src is added


In [12]:
# module for reading data from parquet files in src/data_ingestion.py

from data_ingestion import load_data

In [13]:


df = load_data("../data/raw/yellow_tripdata_2026-01.parquet")
print(df.head())


   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         2  2026-01-01 00:54:04   2026-01-01 00:59:37              1.0   
1         1  2026-01-01 00:34:04   2026-01-01 00:39:47              0.0   
2         1  2026-01-01 00:57:06   2026-01-01 01:05:59              0.0   
3         2  2026-01-01 00:15:22   2026-01-01 00:58:10              4.0   
4         2  2026-01-01 00:27:13   2026-01-01 00:40:43              0.0   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           0.97         1.0                  N           239           238   
1           0.90         1.0                  N           163           162   
2           1.40         1.0                  N            43           237   
3           5.58         1.0                  N           142           209   
4           2.16         1.0                  N            88           144   

   payment_type  fare_amount  extra  mta_tax  tip_amount  tolls_amount  \


In [15]:
# set directory to notebooks. set a relative path to work folder
#os.chdir('C:/Users/Denis Folitse/Desktop/data-science/project-01-urban-mobility/notebooks')
#BASE_DIR = Path('notebooks').resolve().parent
# Path to data

#path_to_data = BASE_DIR/".."/ "data" / "raw"

#ytnyc_path = path_to_data / "yellow_tripdata_2026-01.parquet"
# Check file exists
#if not ytnyc_path.exists():
#    raise FileNotFoundError(f"File not found: {ytnyc_path}")
# read in the data
#tbl = pq.read_table(ytnyc_path)
#df = tbl.to_pandas()

#print(df.head())


In [16]:
# column names
print(df.columns.tolist())

['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']


 Check Data structure


In [26]:
 row, col = df.shape #check the number of rows and columns

print(f"\n Number of rows: {row}")
print(f"\n Number of Columns: {col}")


 Number of rows: 3724889

 Number of Columns: 20


The 2026 January New York TLC trip data has 3,724,889 rows and 20 columns

In [ ]:
df.info() # Checking for the column names and data type

<class 'pandas.DataFrame'>
RangeIndex: 3724889 entries, 0 to 3724888
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     str           
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee            float64   

In [29]:
df.isnull().sum() #check for missing values per column


VendorID                       0
tpep_pickup_datetime           0
tpep_dropoff_datetime          0
passenger_count          1088058
trip_distance                  0
RatecodeID               1088058
store_and_fwd_flag       1088058
PULocationID                   0
DOLocationID                   0
payment_type                   0
fare_amount                    0
extra                          0
mta_tax                        0
tip_amount                     0
tolls_amount                   0
improvement_surcharge          0
total_amount                   0
congestion_surcharge     1088058
Airport_fee              1088058
cbd_congestion_fee             0
dtype: int64

In [ ]:
#df.isna().sum().sum()

np.int64(5440290)

Out of these, 5 columns ( passenger count, RaatecodeID, store and fwd flag, congestion sucharge, airport fee) are all missing 1,088,058 data each. The rest of the columns has a complete data. 
 

## Data validity

In [76]:
# check for:

only_zeros_col = df.columns[(df==0).all()] # columns with only zeros
print("Columns with only zeros:", list(only_zeros_col))

df_without_store_fwdflag = df.drop(columns = ["store_and_fwd_flag","tpep_pickup_datetime","tpep_dropoff_datetime"]) # Pick all columns except store_and_fwd_flag, ,"tpep_pickup_datetime" snf ,"tpep_dropoff_datetime"

#df_without_store_fwdflag.info()

negative_values = df_without_store_fwdflag.columns[(df_without_store_fwdflag<0).any()] # columns that contain negative values
print("Columns containing negative values:", list(negative_values))

#just_fare = (df["fare_amount"]<0).any()
#print(just_fare)

Columns with only zeros: []
Columns containing negative values: ['fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']


From above, we have 10 variables; 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee' that contains negative values. For variables such as fare_amount, tip_amount, tolls_amaiunt, total_amount, etc, it is invalid to have negative values. Hence, this needed to be cleaned. 

In [ ]:
#df["passenger_count"].max()

np.float64(9.0)